# Mfumo wa Wakala wa Microsoft — Azure OpenAI (API za Majibu)

Katika sampuli hii ya msimbo, utatumia **Mfumo wa Wakala wa Microsoft (MAF)** kuunda wakala rahisi unaotegemea **Azure OpenAI** kwa kutumia **API za Majibu**.

> **Kumbuka kuhamia:** Sampuli hii awali ilitumia Semantic Kernel na Miundo ya GitHub. Imepitia uhamishaji kwenda kwa Mfumo wa Wakala wa Microsoft, na Miundo ya GitHub (iliyobadilishwa, itaitimishwa mwezi Julai 2026) imebadilishwa na Azure OpenAI, ambayo inaunga mkono API za Majibu. `OpenAIChatClient` katika MAF inalenga mwisho wa Azure OpenAI wa thabiti `/openai/v1/` na hutumia API za Majibu kwa default.

Kusudi la sampuli hii ni kuonyesha hatua ambazo baadaye zitatumika katika sampuli za msimbo zaidi wakati wa kutekeleza mifumo mbalimbali ya wakala.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Ingiza Vifurushi Vinavyohitajika vya Python


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Kufafanua Zana

Katika Mfumo wa Wakala wa Microsoft, **zana** ni kazi ya kawaida ya Python iliyopambwa na `@tool` ambayo wakala anaweza kuitumia. Hapa chini tunafafanua zana inayorudisha sehemu ya mapumziko ya bahati nasibu na kuepuka kurudia ile ya awali.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Kuunda Wakala

Hapa, tunaunda Wakala anayeitwa `TravelAgent`.

Katika mfano huu, tunatumia maagizo ya msingi sana. Huwezi kubadilisha maagizo haya ili kuona jinsi tabia ya wakala inavyobadilika.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Kuwendesha Wakala

Sasa tunaweza kuendesha wakala. Tunaunda `AgentSession` ili wakala akikumbuke mazungumzo katika mizunguko, kisha tutuma `user_inputs` mbili. Kwanza anaomba safari; ya pili inasema mtumiaji hakupenda pendekezo na anaomba jingine — wakala anatumia historia ya kikao pamoja na zana ya `get_random_destination` kujibu.

Unaweza kubadilisha ujumbe huu ili kuona jinsi wakala anavyotenda tofauti. Majibu hutiririka token kwa token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
